# Modwheel Paper Figure Pack (Colab)

This notebook consolidates the mod30/mod210/mod2310 pre-oracle filtering results into paper/README-ready tables and figures.

It supports the paper-level claim:

> Wheel filters provide a tunable classical pre-oracle reduction layer. Across mod30/mod210/mod2310, candidate streams shrink by about 73--79%; 20news adapter results preserve usable downstream classifier behavior while reducing a fit-time proxy.

Guardrail: these experiments measure adapter behavior and workload proxies. They do **not** claim quantum advantage or accuracy improvement.

In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

## 1. Recreate core wheel density table

In [ ]:
wheel_rows = []
for name, wheel in STANDARD_WHEELS.items():
    wheel_rows.append({
        "wheel": name,
        "primes": "·".join(map(str, wheel.primes)),
        "modulus": wheel.modulus,
        "residues": wheel.residue_count,
        "candidate_fraction": wheel.density,
        "reduction_fraction": wheel.reduction,
    })

wheel_df = pd.DataFrame(wheel_rows)
wheel_df

## 2. Load prior notebook outputs when present

If Notebook 2/3 CSVs have been committed or uploaded into `data/`, this notebook will use them. Otherwise it will create compact fallback tables from known wheel densities.

In [ ]:
synthetic_retained_path = data_dir / "row_id_prefilter_retained_samples.csv"
synthetic_eval_path = data_dir / "row_id_prefilter_classifier_sanity.csv"
news_retained_path = data_dir / "20news_row_id_prefilter_retained_samples.csv"
news_eval_path = data_dir / "20news_row_id_prefilter_svc_sanity.csv"

def fallback_retained(dataset_name, n_original):
    rows = []
    for _, row in wheel_df.iterrows():
        n_retained = int(round(n_original * row["candidate_fraction"]))
        rows.append({
            "dataset": dataset_name,
            "subset": row["wheel"],
            "n_original": n_original,
            "n_retained": n_retained,
            "retained_fraction": n_retained / n_original,
            "reduction_fraction": 1 - n_retained / n_original,
        })
    return pd.DataFrame(rows)

if synthetic_retained_path.exists():
    synthetic_retained = pd.read_csv(synthetic_retained_path)
    synthetic_retained["dataset"] = "synthetic_adapter"
else:
    synthetic_retained = fallback_retained("synthetic_adapter", 6000)

if news_retained_path.exists():
    news_retained = pd.read_csv(news_retained_path)
    news_retained["dataset"] = "20news"
else:
    # Fallback uses the observed Notebook 3 baseline from the run: 3729 samples.
    news_retained = fallback_retained("20news", 3729)

if news_eval_path.exists():
    news_eval = pd.read_csv(news_eval_path)
else:
    # Fallback values from Notebook 3 run shared in chat.
    news_eval = pd.DataFrame([
        {"subset": "baseline_all_rows", "n_samples": 3729, "fit_time_seconds": 0.0478, "balanced_accuracy": 0.854},
        {"subset": "mod30", "n_samples": 994, "fit_time_seconds": 0.0155, "balanced_accuracy": 0.854},
        {"subset": "mod210", "n_samples": 852, "fit_time_seconds": 0.0130, "balanced_accuracy": 0.877},
        {"subset": "mod2310", "n_samples": 775, "fit_time_seconds": 0.0127, "balanced_accuracy": 0.840},
    ])

synthetic_retained.head(), news_retained.head(), news_eval

## 3. Build consolidated summary tables

In [ ]:
# Normalize retained tables to a shared schema.
def normalize_retained(df, dataset_name):
    out = df.copy()
    if "wheel" in out.columns and "subset" not in out.columns:
        out = out.rename(columns={"wheel": "subset"})
    out["dataset"] = dataset_name
    keep = ["dataset", "subset", "n_original", "n_retained", "retained_fraction", "reduction_fraction"]
    return out[keep]

retained_summary = pd.concat([
    normalize_retained(synthetic_retained, "synthetic_adapter"),
    normalize_retained(news_retained, "20news"),
], ignore_index=True)

# Add a compact wheel-only summary.
wheel_export = wheel_df.copy()

# Attach 20news behavior metrics when available.
news_behavior = news_eval.copy()
if "accuracy" not in news_behavior.columns:
    news_behavior["accuracy"] = np.nan

paper_summary_csv = data_dir / "paper_results_summary.csv"
retained_summary_csv = data_dir / "paper_retained_fraction_summary.csv"
news_behavior_csv = data_dir / "paper_20news_behavior_summary.csv"

wheel_export.to_csv(paper_summary_csv, index=False)
retained_summary.to_csv(retained_summary_csv, index=False)
news_behavior.to_csv(news_behavior_csv, index=False)

wheel_export, retained_summary, news_behavior

## 4. Figure 1 — Wheel density / candidate fraction

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(wheel_df["wheel"], wheel_df["candidate_fraction"], marker="o", linewidth=2)
ax.set_title("Wheel-Based Pre-Oracle Filtering: Candidate Fraction")
ax.set_xlabel("Wheel filter")
ax.set_ylabel("Remaining candidate fraction")
ax.set_ylim(0, 0.32)
ax.grid(True, alpha=0.35)

for _, row in wheel_df.iterrows():
    label = f"{row['residues']}/{row['modulus']} = {row['candidate_fraction']:.1%}\n{row['reduction_fraction']:.1%} excluded"
    ax.annotate(label, (row["wheel"], row["candidate_fraction"]), textcoords="offset points", xytext=(0, 10), ha="center")

fig.tight_layout()
fig1_svg = fig_dir / "figure_01_wheel_candidate_fraction.svg"
fig1_png = fig_dir / "figure_01_wheel_candidate_fraction.png"
fig.savefig(fig1_svg)
fig.savefig(fig1_png, dpi=220)
plt.show()
fig1_svg, fig1_png

## 5. Figure 2 — Adapter retained fractions: synthetic vs 20news

In [ ]:
plot_retained = retained_summary.copy()
order = ["mod30", "mod210", "mod2310"]
x = np.arange(len(order))
width = 0.36

synthetic_vals = plot_retained[plot_retained["dataset"] == "synthetic_adapter"].set_index("subset").loc[order, "retained_fraction"]
news_vals = plot_retained[plot_retained["dataset"] == "20news"].set_index("subset").loc[order, "retained_fraction"]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(x - width/2, synthetic_vals, width, label="synthetic adapter")
ax.bar(x + width/2, news_vals, width, label="20news")
ax.set_title("Row-ID Prefilter Adapter: Retained Sample Fraction")
ax.set_xlabel("Wheel filter")
ax.set_ylabel("Retained sample fraction")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylim(0, 0.32)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()

for i, value in enumerate(synthetic_vals):
    ax.text(i - width/2, value + 0.008, f"{value:.1%}", ha="center", fontsize=9)
for i, value in enumerate(news_vals):
    ax.text(i + width/2, value + 0.008, f"{value:.1%}", ha="center", fontsize=9)

fig.tight_layout()
fig2_svg = fig_dir / "figure_02_adapter_retained_fraction.svg"
fig2_png = fig_dir / "figure_02_adapter_retained_fraction.png"
fig.savefig(fig2_svg)
fig.savefig(fig2_png, dpi=220)
plt.show()
fig2_svg, fig2_png

## 6. Figure 3 — 20news behavior: balanced accuracy and fit-time proxy

In [ ]:
behavior = news_behavior.copy()
subset_order = [s for s in ["baseline_all_rows", "mod30", "mod210", "mod2310"] if s in behavior["subset"].tolist()]
behavior = behavior.set_index("subset").loc[subset_order].reset_index()

# Save separate paper-safe SVGs rather than using dual axes.
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(behavior["subset"], behavior["balanced_accuracy"], marker="o", linewidth=2)
ax.set_title("20news Downstream Behavior After Row-ID Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
plt.xticks(rotation=20)
fig.tight_layout()
fig3a_svg = fig_dir / "figure_03a_20news_balanced_accuracy.svg"
fig3a_png = fig_dir / "figure_03a_20news_balanced_accuracy.png"
fig.savefig(fig3a_svg)
fig.savefig(fig3a_png, dpi=220)
plt.show()

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(behavior["subset"], behavior["fit_time_seconds"])
ax.set_title("20news Fit-Time Proxy After Row-ID Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Fit time (seconds)")
ax.grid(True, axis="y", alpha=0.35)
plt.xticks(rotation=20)
fig.tight_layout()
fig3b_svg = fig_dir / "figure_03b_20news_fit_time_proxy.svg"
fig3b_png = fig_dir / "figure_03b_20news_fit_time_proxy.png"
fig.savefig(fig3b_svg)
fig.savefig(fig3b_png, dpi=220)
plt.show()

fig3a_svg, fig3b_svg

## 7. Export Markdown summary for README/paper draft

In [ ]:
summary_md = data_dir / "paper_results_summary.md"

md = []
md.append("# Modwheel Pre-Oracle Filtering Results Summary\n")
md.append("## Wheel density\n")
md.append(wheel_df.to_markdown(index=False))
md.append("\n\n## Retained sample fractions\n")
md.append(retained_summary.to_markdown(index=False))
md.append("\n\n## 20news downstream behavior\n")
md.append(news_behavior.to_markdown(index=False))
md.append("\n\n## Paper-ready interpretation\n")
md.append(
    "Wheel filters define a tunable classical pre-oracle reduction layer. "
    "For mod30, mod210, and mod2310, candidate streams are reduced by roughly 73--79%. "
    "In the 20news adapter experiment, row-ID filtering reduces sample count and fit-time proxy while preserving usable downstream classifier behavior. "
    "These results support adapter-level integration before QOS sampling/oracle construction, without modifying QOS internals."
)
md.append("\n\nGuardrail: these experiments measure filtering behavior and workload proxies; they do not claim quantum advantage or accuracy improvement.\n")

summary_md.write_text("\n".join(md), encoding="utf-8")
print(summary_md.read_text())

## 8. Download outputs

In [ ]:
from google.colab import files

download_paths = [
    "data/paper_results_summary.csv",
    "data/paper_retained_fraction_summary.csv",
    "data/paper_20news_behavior_summary.csv",
    "data/paper_results_summary.md",
    "figures/figure_01_wheel_candidate_fraction.svg",
    "figures/figure_02_adapter_retained_fraction.svg",
    "figures/figure_03a_20news_balanced_accuracy.svg",
    "figures/figure_03b_20news_fit_time_proxy.svg",
]

for path in download_paths:
    files.download(path)